In [ ]:
from google.colab import drive
import pandas as pd

# 1️⃣ 구글 드라이브 마운트
drive.mount('/content/drive')

# 2️⃣ 파일 경로 지정 (파일명과 폴더 확인)
base_path = '/content/drive/MyDrive/'  # MyDrive 바로 아래에 파일이 있다면 이 경로

train_path = base_path + 'train_ad.csv'
test_path = base_path + 'test_ad.csv'
sample_path = base_path + 'sample_submission_ad.csv'

# 3️⃣ 데이터 읽기
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sample_path)

# 4️⃣ 데이터 확인
print(train.head())
print(test.head())
print(submission.head())

In [ ]:
from xgboost import XGBClassifier, plot_importance
import matplotlib.pyplot as plt
import pandas as pd

# --- 데이터 준비 ---
X_train = train.drop(columns=['target', 'ID'])
y_train = train['target']

# --- XGBoost 모델 학습 ---
xgb_model = XGBClassifier(
    random_state=42,
    objective='multi:softprob',  # 다중 클래스
    eval_metric='mlogloss',
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train)

# --- Feature Importance 추출 (모든 feature) ---
importance_dict = xgb_model.get_booster().get_score(importance_type='weight')
importance_df = pd.DataFrame({
    'feature': list(importance_dict.keys()),
    'importance': list(importance_dict.values())
}).sort_values(by='importance', ascending=False)

# --- 순서대로 출력 ---
print("=== Feature Importance (전체) ===")
print(importance_df)

# --- CSV 저장 ---
importance_df.to_csv('xgb_feature_importance.csv', index=False)
print("Feature importance CSV 저장 완료: xgb_feature_importance.csv")

# --- 시각화 (전체) ---
plt.figure(figsize=(12, 10))
plot_importance(xgb_model)  # max_num_features 생략하면 전체
plt.show()



In [ ]:
import shap
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train)

# 전체 feature importance
shap.summary_plot(shap_values, X_train)

# 특정 샘플 분석
shap.force_plot(explainer.expected_value[0], shap_values[0][0,:], X_train.iloc[0,:])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# --- PCA 2차원 적용 ---
features = train.drop(columns=['ID', 'target'])
target = train['target']

pca = PCA(n_components=2)
pca_result = pca.fit_transform(features)

# --- 3D 시각화 ---
fig = plt.figure(figsize=(12,8))
ax = fig.add_subplot(111, projection='3d')

# 색상 맵 생성 (타겟이 0~20이면 cmap='tab20' 추천)
scatter = ax.scatter(
    pca_result[:,0],
    pca_result[:,1],
    target,
    c=target,
    cmap='tab20',
    s=50,
    alpha=0.8
)

# 축 레이블
ax.set_xlabel('PCA 1', fontsize=12)
ax.set_ylabel('PCA 2', fontsize=12)
ax.set_zlabel('Target', fontsize=12)
ax.set_title('PCA 1 vs PCA 2 vs Target', fontsize=14)

# 컬러바 표시
cbar = fig.colorbar(scatter, ax=ax, pad=0.1, shrink=0.5)
cbar.set_label('Target', rotation=270, labelpad=15)

plt.show()


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
import plotly.express as px

# --- 데이터 준비 ---
features = train.drop(columns=['ID', 'target'])
target = train['target']

# --- PCA 적용 ---
pca = PCA(n_components=2)
pca_result = pca.fit_transform(features)

pca_df = pd.DataFrame({
    'PCA1': pca_result[:,0],
    'PCA2': pca_result[:,1],
    'Target': target.astype(str)  # 문자열로 변환 (범주형)
})

# --- 21가지 색상 만들기 ---
# Plotly qualitative color scale 중 Safe + Pastel1 + Dark24 등 섞어서 사용 가능
colors = px.colors.qualitative.Alphabet  # 26가지 색상 있음, 21개 커버 가능
color_map = {str(i): colors[i] for i in range(21)}  # 0~20 매핑

# --- Plotly 3D Scatter Plot ---
fig = px.scatter_3d(
    pca_df,
    x='PCA1',
    y='PCA2',
    z='Target',
    color='Target',
    color_discrete_map=color_map,
    opacity=0.8,
    title='PCA 1 vs PCA 2 vs Target'
)

fig.update_layout(
    scene=dict(
        xaxis_title='PCA 1',
        yaxis_title='PCA 2',
        zaxis_title='Target'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()


In [ ]:
train.shape

In [ ]:
import matplotlib.pyplot as plt

features = train.drop(columns=['ID', 'target'])
target = train['target']

num_features = len(features.columns)
cols = 4  # 한 행에 4개 subplot
rows = (num_features + cols - 1) // cols  # 필요한 행 계산

plt.figure(figsize=(20, rows*4))

for i, col in enumerate(features.columns, 1):
    plt.subplot(rows, cols, i)
    plt.scatter(features[col], target, alpha=0.6)
    plt.xlabel(col)
    plt.ylabel('Target')
    plt.title(f'{col} vs Target')

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

# --- feature 간 상관계수 계산 ---
corr_matrix = features.corr()

# --- 히트맵 그리기 ---
plt.figure(figsize=(16,12))
sns.heatmap(corr_matrix,
            annot=False,      # 숫자 표시 여부
            cmap='coolwarm',  # 색상 맵
            center=0,         # 0 기준으로 색상 변화
            linewidths=0.5)   # 격자 선 두께
plt.title('Feature Correlation Heatmap', fontsize=16)
plt.show()


In [ ]:
import pandas as pd

# 모든 행 출력
pd.set_option('display.max_rows', None)

# 모든 열 출력 (필요 시)
pd.set_option('display.max_columns', None)

# 소수점 자리수 제한 해제 (선택)
pd.set_option('display.float_format', lambda x: '%.5f' % x)

# 다중공선성 의심 변수 쌍 출력
threshold = 0.8
corr_pairs = corr_matrix.abs().unstack()
corr_pairs = corr_pairs[(corr_pairs < 1.0) & (corr_pairs >= threshold)]
corr_pairs = corr_pairs.sort_values(ascending=False)

print("=== 다중공선성 의심 변수 쌍 (|상관계수| >= 0.8) ===")
print(corr_pairs)


In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np
import matplotlib.pyplot as plt

features = train.drop(columns=['ID', 'target'])
target = train['target']
# --- 상관계수 계산 ---
corr_matrix = features.corr().abs()

# --- 상관이 높은 변수 그룹 찾기 (threshold = 0.8) ---
threshold = 0.8
corr_matrix_unstack = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(i, j) for i, j in zip(*np.where(corr_matrix_unstack >= threshold))]

# 변수 이름으로 변환
high_corr_pairs_names = [(corr_matrix.columns[i], corr_matrix.columns[j]) for i,j in high_corr_pairs]

# --- 그룹 단위로 묶기 ---
groups = []
for v1, v2 in high_corr_pairs_names:
    added = False
    for g in groups:
        if v1 in g or v2 in g:
            g.update([v1, v2])
            added = True
            break
    if not added:
        groups.append(set([v1,v2]))

# --- VIF 계산 함수 ---
def compute_vif(df):
    X = df.copy()
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

# --- 그룹 내에서 VIF가 가장 높은 변수 제거 ---
features_to_keep = set(features.columns)
for g in groups:
    if len(g) <= 1:
        continue
    g_vif = compute_vif(features[list(g)])
    drop_var = g_vif.sort_values('VIF', ascending=False).iloc[0]['feature']
    features_to_keep.discard(drop_var)
    print(f"Group {g} → 제거 변수: {drop_var}")

# --- 최종 학습용 feature 리스트 ---
features_cleaned = features[list(features_to_keep)]
print(f"\n최종 학습용 feature 수: {features_cleaned.shape[1]}")

In [ ]:
# 선택 변수
vars_to_print = ['X_07', 'X_33']

# 해당 컬럼 출력
print(features[vars_to_print])

In [ ]:
var1 = 'X_07'
var2 = 'X_33'

corr_value = features[var1].corr(features[var2])
print(f"{var1}와 {var2}의 상관계수: {corr_value:.4f}")


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

# feature 데이터 준비
X = features.copy()

# VIF 계산
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# VIF 높은 순으로 정렬
vif_data = vif_data.sort_values(by="VIF", ascending=False)
print(vif_data)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for col in features.columns:
    plt.figure(figsize=(6,3))
    sns.histplot(features[col], kde=True)
    plt.title(f'{col} distribution')
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for col in features.columns:
    plt.figure(figsize=(12,4))
    sns.boxplot(x=target, y=features[col])
    plt.title(f'{col} vs Target')
    plt.xlabel('Target')
    plt.ylabel(col)
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# feature 데이터
features = train.drop(columns=['ID', 'target'])
target = train['target']

# feature와 target 간 상관계수 계산
corr_with_target = features.corrwith(target)

# 출력
print(corr_with_target.sort_values(ascending=False))

# 시각화
plt.figure(figsize=(12,6))
sns.barplot(x=corr_with_target.index, y=corr_with_target.values)
plt.xticks(rotation=90)
plt.ylabel("Correlation with Target")
plt.title("Feature vs Target Correlation")
plt.show()


In [ ]:
import numpy as np

# 새로운 feature 생성: X_33 제곱 후 루트
train['X_33_sqrt'] = np.sqrt(np.sqrt(np.sqrt(train['X_28']))) # 절댓값 루트
# 만약 음수값이 포함될 수 있으면 np.sqrt(np.abs(...)) 사용

# 타겟과의 상관관계 확인
corr_with_target = train[['X_33_sqrt', 'target']].corr()['target'].drop('target')
print(corr_with_target)

# 시각화
import matplotlib.pyplot as plt
plt.scatter(train['X_33_sqrt'], train['target'], alpha=0.6)
plt.xlabel('sqrt(X_33^2)')
plt.ylabel('Target')
plt.title('Correlation with Target')
plt.show()


In [ ]:
import shap

# 트리 기반 모델이면 TreeExplainer 사용
explainer = shap.TreeExplainer(xgb_model)

# 전체 train 데이터에 대한 shap value 계산
shap_values = explainer.shap_values(X_train)

# 1️⃣ Global: 전체 feature importance
shap.summary_plot(shap_values, X_train)

# 2️⃣ Local: 특정 샘플
#shap.force_plot(explainer.expected_value[0], shap_values[0][0,:], X_train.iloc[0,:])
